# Inspect terrain and patch readiness

**Historical investigation, not current terrain coverage.** The original Earth tile/halo defect was corrected in [notebook 09](09_verify_earth_coverage_fix.ipynb). Lunar source heights were finite; our image-support mask introduced the elevation holes. [Notebook 13](13_build_lunar_terrain_views.ipynb) supplies a full-context view with zero elevation gaps inside the seven ROIs. Read [notebook 14](../exploration/14_terrain_feature_eda.ipynb) for the corrected terrain and separate support views. Remaining derivative edge exclusions and genuine environmental missingness are different issues.

**Question:** Which cells and patches passed the historical strict-support policy?

Run all cells in order using the project environment. Reads the corrected Earth run and unchanged Moon run; never modifies them. Saves tables, PNG maps, SVG charts, input hashes and an actionable report under `data/processed/analysis/`. No downloads or package installation.

Counts are exact for the chosen upper-left-anchored, nonoverlapping lattice. They are not all possible placements or independent training samples. 300/600/960 m are exploratory scenarios, not an approved model configuration. Different scenarios overlap one another and must not be pooled as independent examples.

**Workflow stage: preprocessing / data readiness.** Use this notebook to inspect coverage, investigate exclusions, or verify corrections before EDA. Its saved results and pinned inputs preserve the original investigation. Notebook 13 builds the two lunar terrain views; notebook 14 explores the corrected Earth run and those views. See the [workflow and acceptance checks](../README.md). Moving this notebook does not change when the original findings were made.


In [ ]:
PREFIX = 'terrain-inspection-'
TITLE = 'Terrain inspection: conclusions and actions'
NOTEBOOK = 'preprocessing/10_inspect_terrain_and_patch_readiness.ipynb'
import json
import sys
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from IPython.display import SVG, Image, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT))
from notebooks.shared.inspection import bar_svg, check_input, complete_blocks, file_hash, verify_run

OUT = ROOT / 'data/processed/analysis' / (PREFIX + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
evidence = {}
expected = {}
actions = []

def register(stage, run_id):
    run = ROOT / 'data/processed' / stage / run_id
    expected.update(verify_run(run))
    evidence[str(run / 'provenance.json')] = file_hash(run / 'provenance.json')
    return run

def checked(path):
    return check_input(path, expected, evidence)

def read_layer(path):
    with rasterio.open(checked(path)) as src:
        assert src.count == 1 and src.scales == (1.0,) and src.offsets == (0.0,), path
        return src.read(1, masked=True).astype('float32').filled(np.nan), (src.shape, src.transform, src.crs)

def chart(labels, values, title, unit, filename):
    svg = bar_svg(list(labels), [float(v) for v in values], title, unit)
    (OUT / filename).write_text(svg, encoding='utf-8')
    display(SVG(svg))


In [ ]:
PATCH_SIDES_M = [300, 600, 960]
FEATURES = ['elevation_m', 'relative_elevation_m', 'slope_deg', 'laplacian_per_m', 'local_elevation_std_m']
MIN_IMAGE_SUPPORT = 0.99
CONFIG = {'patch_sides_m': PATCH_SIDES_M, 'features': FEATURES, 'minimum_image_support': MIN_IMAGE_SUPPORT,
          'lattice': 'upper-left anchored, nonoverlapping; remainder strips omitted'}
assert FEATURES and len(set(FEATURES)) == len(FEATURES)
assert all(isinstance(s, int) and s > 0 and s % 30 == 0 for s in PATCH_SIDES_M)
assert 0 < MIN_IMAGE_SUPPORT <= 1
earth = register('earth_terrain_landcover', '20261028T063142045472Z_9d6d3b')
moon = register('lunar_terrain_environment', '20261028T022855014566Z_cffea7')
folders = [('earth', p) for p in sorted((earth / 'earth').glob('*/grid_30m'))]
folders += [('moon', p) for p in sorted((moon / 'moon').glob('*/grid_30m'))]
assert len(folders) == 10
print('Regions:', len(folders), '| channels:', FEATURES, '| patch sides (m):', PATCH_SIDES_M)


## Historical eligibility masks, distributions, and complete patches

Map colors: gray = outside ROI; orange = screened out/unsupported; red = passes screen/support but missing a selected feature; green = eligible. Maps are decimated nearest-neighbor previews (top is north); numerical summaries use every full-resolution cell. Detailed patch JSON records row/column and projected bounds in the region CRS. Quantiles are over eligible cells, not all terrain, and can reflect screening bias.

Standalone elevation/slope previews from the old masked lunar view have been removed. The retained categorical maps explain the historical selection policy; orange/red areas do not mean source elevation is absent. Current terrain maps are in notebook 14.

In [ ]:
coverage_rows, patch_rows, distribution_rows = [], [], []
for body, folder in folders:
    region = folder.parent.name
    mask_name = 'natural_screen_mask' if body == 'earth' else 'image_support_fraction'
    mask, grid = read_layer(folder / (mask_name + '.tif'))
    shape, transform, crs = grid
    assert transform.a == 30 and transform.e == -30 and transform.b == transform.d == 0
    inside = np.isfinite(mask)
    selected = inside & (mask == 1) if body == 'earth' else inside & (mask >= MIN_IMAGE_SUPPORT)
    joint = inside.copy()
    for feature in FEATURES:
        values, feature_grid = read_layer(folder / (feature + '.tif'))
        assert feature_grid == grid, (region, feature)
        joint &= np.isfinite(values)
    eligible = selected & joint
    coverage_rows.append({'body': body, 'region': region, 'roi_cells': int(inside.sum()),
                          'selected_cells': int(selected.sum()), 'eligible_cells': int(eligible.sum()),
                          'missing_feature_cells_after_selection': int((selected & ~joint).sum()),
                          'eligible_area_km2': float(eligible.sum() * 0.0009)})
    for feature in FEATURES:
        values, _ = read_layer(folder / (feature + '.tif'))
        valid = values[eligible]
        if valid.size:
            quantiles = np.quantile(valid, [0, .01, .5, .99, 1])
            distribution_rows.append(dict(region=region, feature=feature, count=len(valid),
                **dict(zip(['min', 'p01', 'median', 'p99', 'max'], map(float, quantiles), strict=True))))
    for metres in PATCH_SIDES_M:
        side = metres // 30
        blocks = complete_blocks(eligible, side)
        candidates = complete_blocks(inside, side)
        coords = np.argwhere(blocks)
        locations = []
        for block_row, block_col in coords:
            row, col = int(block_row * side), int(block_col * side)
            left, top = transform * (col, row)
            right, bottom = transform * (col + side, row + side)
            locations.append({'row': row, 'col': col, 'bounds': [left, bottom, right, top]})
        (OUT / f'{region}_{metres}m_patches.json').write_text(json.dumps({
            'crs_wkt': crs.to_wkt(), 'side_cells': side, 'patches': locations}), encoding='utf-8')
        patch_rows.append({'region': region, 'side_m': metres, 'complete_roi_candidates': int(candidates.sum()),
                           'eligible_patches': int(blocks.sum()),
                           'remainder_cells': int(inside.size - blocks.size * side * side)})
    category = np.where(~inside, 0, np.where(~selected, 1, np.where(~joint, 2, 3)))
    stride = max(1, int(np.ceil(max(shape) / 650)))
    colors = np.array([[210, 210, 210], [230, 170, 70], [190, 45, 65], [35, 135, 100]], dtype='uint8')
    rgb = colors[category[::stride, ::stride]]
    preview = OUT / f'{region}_eligibility.png'
    with rasterio.open(preview, 'w', driver='PNG', width=rgb.shape[1], height=rgb.shape[0], count=3,
                       dtype='uint8') as dst:
        dst.write(rgb.transpose(2, 0, 1))
    display(Markdown(f'### {region} ({body})'))
    display(Image(filename=str(preview), width=550))
    del mask, values, category, rgb
coverage = pd.DataFrame(coverage_rows)
patches = pd.DataFrame(patch_rows)
distributions = pd.DataFrame(distribution_rows)
coverage.to_json(OUT / 'coverage.json', orient='records', indent=2)
patches.to_json(OUT / 'patch_counts.json', orient='records', indent=2)
distributions.to_json(OUT / 'feature_quantiles.json', orient='records', indent=2)
display(coverage)
display(distributions)
display(patches.pivot(index='region', columns='side_m', values='eligible_patches'))


## Decisions supported by these results

Positive patch counts establish computational availability only. Reject a scenario/region with zero complete patches; otherwise review full-resolution candidate patches and assign spatial groups before selecting a model input size. No outlier is removed solely for being extreme. Compare the 1st/99th percentiles and extrema in the feature table before fitting any normalization.

In [ ]:
for metres in PATCH_SIDES_M:
    subset = patches.loc[patches.side_m == metres]
    chart(subset.region, subset.eligible_patches, f'Complete {metres} m patches', 'patch count', f'patches_{metres}m.svg')
    empty = subset.loc[subset.eligible_patches == 0, 'region'].tolist()
    actions.append(f'{metres} m: {int(subset.eligible_patches.sum())} complete patches across regions. '
                   + (f'Exclude zero-count regions from this scenario: {empty}.' if empty else
                      'All regions have complete patches; review local artifacts and spatial splits before extraction.'))
for row in coverage.to_dict('records'):
    actions.append(f"{row['region']}: {row['eligible_area_km2']:.3f} km² eligible; "
                   f"{row['missing_feature_cells_after_selection']} selected cells excluded for missing features. "
                   'Preserve the complete-feature intersection.')
actions.append('Retain Earth land-cover exclusions and lunar image-support exclusions. Do not fill gaps to increase patch counts.')
actions.append('Patch counts depend on origin, size and channels. Neighboring patches remain spatially correlated; no train/test split is created here.')
# Commit a completion marker only after all checks and exports succeed.
report = '# ' + TITLE + '\n\n' + '\n'.join('- ' + item for item in actions)
report += '\n\nThese are data-readiness conclusions, not scientific accuracy or site-safety certification. '
report += 'Inputs were read-only. Consult the tables and figures in this folder for measurements.\n'
(OUT / 'REPORT.md').write_text(report, encoding='utf-8')
evidence[str(ROOT / 'notebooks' / NOTEBOOK)] = file_hash(ROOT / 'notebooks' / NOTEBOOK)
evidence[str(ROOT / 'notebooks/shared/inspection.py')] = file_hash(ROOT / 'notebooks/shared/inspection.py')
(OUT / 'provenance.json').write_text(json.dumps({
    'status': 'publishing', 'inputs_sha256': evidence, 'configuration': CONFIG,
    'versions': {'python': sys.version, 'numpy': np.__version__, 'pandas': pd.__version__, 'rasterio': rasterio.__version__},
    'outputs_sha256': {p.name: file_hash(p) for p in OUT.iterdir() if p.is_file()},
}, indent=2), encoding='utf-8')
display(Markdown(report))
print('Saved:', OUT)


## Save reusable data and disposable reports
Run this final cell after the displays. Later notebooks read only the data directory; reports can be deleted and regenerated.

In [ ]:
import sys

sys.path.insert(0, str(ROOT))
from notebooks.shared.analysis_artifacts import publish_analysis

REPORT_OUT = publish_analysis(ROOT, OUT, manifest_name="provenance.json")
print("Reusable data:", OUT)
print("Disposable reports:", REPORT_OUT)